# Reproducibility Notebook

This notebook runs the reproducibility package for the dissertation *Impact of Large Language Models on Phishing Email Detection: A Comparative Analysis of Classical Machine Learning and Deep Learning Models* in a few clicks.

Run the **Setup** cell first, then any option below.

| Option | What it does | Time | Needs |
|---|---|---|---|
| **A. View all results** | Prints every Chapter 4 result and shows the results page | Seconds | Python only |
| **Check the files** | Confirms every file matches its recorded checksum | Seconds | Python only |
| **B. Quick test rerun** | Runs the whole pipeline with 2 iterations per analysis, to check that it works | About 15 minutes | pandas, NumPy, SciPy, scikit-learn |
| **C. Full rerun** | Reruns the study from the raw data | Several hours | Same, plus an NVIDIA GPU for RQ2 |
| **D. One stage at a time** | Runs the dataset rebuild, RQ1, RQ2, RQ3, or the outlier check separately | Minutes to hours per stage | Same as C |

## Setup

If this notebook was opened on its own (for example, in Google Colab), this cell downloads the package from GitHub first. Otherwise it only confirms the working folder.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO = "https://github.com/Guillaume-star/PhD-Dissertation-Data-Analysis-Reproducibility-Package.git"
if not Path("print_results.py").exists():
    folder = Path("PhD-Dissertation-Data-Analysis-Reproducibility-Package")
    if not folder.exists():
        subprocess.run(["git", "clone", REPO], check=True)
    os.chdir(folder)
print("Working in:", Path.cwd())

## Option A: View all results

Prints RQ1 to RQ3, Tables 1 and 2, the four sensitivity analyses, AUC-ROC, and the outlier check, read directly from the output files in `results/`. It also confirms that the final datasets match their frozen checksums and saves `RESULTS_SUMMARY.html`.

In [ ]:
%run print_results.py

The same results as a formatted page:

In [ ]:
from IPython.display import HTML
HTML(Path("RESULTS_SUMMARY.html").read_text(encoding="utf-8"))

## Check the files

Compares every file with its SHA-256 checksum in `CHECKSUMS_SHA256.txt`. The raw Enron archive is listed there but not stored on GitHub (it is too large), so it is reported as not included. Option C downloads it when needed.

In [ ]:
import hashlib

matched = differ = not_included = 0
for line in Path("CHECKSUMS_SHA256.txt").read_text(encoding="utf-8").splitlines():
    expected, name = line.split("  ", 1)
    path = Path(name)
    if not path.exists():
        not_included += 1
        print("not included:", name)
    elif hashlib.sha256(path.read_bytes()).hexdigest() == expected:
        matched += 1
    else:
        differ += 1
        print("DIFFERENT:", name)
print(f"{matched} files match, {differ} differ, {not_included} not included")

## Option B: Quick test rerun

Checks that the whole pipeline runs on this computer: it rebuilds the dataset from the raw files (downloading the raw Enron archive the first time, about 423 MB) and runs every analysis with 2 iterations. **The numbers it prints are not the study's results**, because 2 iterations are far too few.

First, install the required libraries if they are missing. If `pip` refuses with an "externally managed environment" message, choose a kernel from a virtual environment or a conda environment, or run this notebook in Google Colab.

In [ ]:
import importlib.util

needed = {"pandas": "pandas", "numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn"}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    %pip install -r requirements.txt
else:
    print("Required libraries are already installed.")

Run the quick test. Outputs go to `rerun_workspace/` inside this folder; the package files are never changed. RQ2 runs only if an NVIDIA GPU is available; otherwise it is skipped with instructions.

In [ ]:
%run run_all.py --quick --workspace rerun_workspace

## Option C: Full rerun

Reruns the study with the same settings as the reported results (500 iterations for the primary analysis and 100 for each sensitivity analysis). Run the library check in Option B first.

- **Time:** about 1.5 hours for RQ1's primary analysis and 20 minutes for each sensitivity analysis on a CPU.
- **RQ2 (deep learning)** needs an NVIDIA GPU. Without one, run RQ2 on Google Colab with `RQ2_on_Colab.ipynb`, which explains where to put its outputs, then run the RQ3 and outlier stages.
- When it finishes, every rerun result is printed next to the reported one ("match" or "differs") and saved as `RERUN_RESULTS_SUMMARY.html` in the workspace.

To run RQ3 and the outlier check without a GPU, using the reported RQ2 outputs, add `--use-reported-rq2` to the command.

In [ ]:
%run run_all.py --workspace rerun_workspace

## Option D: Run one stage at a time

Each cell below runs one stage of the full rerun, so the research questions can be checked separately. Run the library check in Option B first. Every stage writes to the same `rerun_workspace/` folder, and each RQ stage runs the primary analysis and the four sensitivity analyses, then prints its results next to the reported ones.

To make any stage a fast check instead (2 iterations, results not expected to match), add `--quick` to its command. Do not mix `--quick` and full runs in the same workspace.

### Rebuild the dataset (optional)

Rebuilds the final dataset from the raw files and confirms it matches the frozen version. Downloads the raw Enron archive (423 MB) the first time. RQ1 and RQ2 can run without this step: they then use the frozen final dataset in `data/final/`.

In [ ]:
%run run_all.py --stage data --workspace rerun_workspace

### RQ1: Classical machine learning

Logistic Regression, Linear SVM, and Random Forest on a CPU. About 1.5 hours for the primary analysis (500 iterations) and 20 minutes for each sensitivity analysis.

In [ ]:
%run run_all.py --stage rq1 --workspace rerun_workspace

### RQ2: Deep learning

LSTM, GRU, and CNN. Needs an NVIDIA GPU; without one, this cell prints instructions for running RQ2 on Google Colab with `RQ2_on_Colab.ipynb`.

In [ ]:
%run run_all.py --stage rq2 --workspace rerun_workspace

Without a GPU, run this cell instead. It copies the reported RQ2 outputs from `results/` into the workspace (this is not a rerun of RQ2), so that RQ3 and the outlier check can run.

In [ ]:
%run run_all.py --stage rq2 --use-reported-rq2 --workspace rerun_workspace

### RQ3: Comparison of model families

Compares the change in discriminability between the two model families. Needs the RQ1 and RQ2 outputs in the workspace (run the two stages above first). Takes a few seconds.

In [ ]:
%run run_all.py --stage rq3 --workspace rerun_workspace

### Outlier check

Reruns the primary-analysis tests after removing extreme difference scores (1.5 × IQR rule and ±3 SD rule). Needs the RQ3 outputs. Takes a few seconds.

In [ ]:
%run run_all.py --stage outlier --workspace rerun_workspace